# Import & Data Loading

In [5]:
import pandas as pd

df = pd.read_csv('RetailCleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'],utc=True).dt.date
print(df.shape)
df.head()
df.tail()

/var/folders/q2/s9rksbg53mn8bqfg7nh2_lrc0000gn/T/ipykernel_22032/738917586.py:3: DtypeWarning: Columns (0: Invoice) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('RetailCleaned.csv')


(1033036, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet
1033031,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09,0.83,14397.0,United Kingdom,2010-2011
1033032,C581499,M,Manual,-1,2011-12-09,224.69,15498.0,United Kingdom,2010-2011
1033033,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09,10.95,15311.0,United Kingdom,2010-2011
1033034,C581569,20979,36 PENCILS TUBE RED RETROSPOT,-5,2011-12-09,1.25,17315.0,United Kingdom,2010-2011
1033035,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09,1.25,17315.0,United Kingdom,2010-2011


# Recency, Frequency & Monetary

In [8]:
rfm_data = df[
    (df['Customer_ID'].notna())&
    (df['Quantity']>0)
].copy()

snape_date = pd.to_datetime('2011-12-10').date()

rfm = rfm_data.groupby('Customer_ID').agg(
    Recency = ('InvoiceDate', lambda x:(snape_date - x.max()).days),
    Frequency = ('Invoice','nunique'),
    Monetary = ('Quantity', lambda x:(x*rfm_data.loc[x.index,'Price']).sum())
).reset_index()

# Segmentation

In [13]:
rfm['R_Score'] = pd.qcut(rfm['Recency'],5,labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),5,labels=[5,4,3,2,1]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],5,labels=[5,4,3,2,1]).astype(int)

def segment_customer(row):
    if row['R_Score'] >=4 and row['F_Score'] >=4 and row['M_Score'] >=4:
        return 'Champions'
    elif row['R_Score'] >=3 and row['F_Score'] >=3:
        return 'Loyal'
    elif row['R_Score'] <=2 and row['F_Score'] >=3:
        return 'At Risk'
    elif row['R_Score'] >=4 and row['F_Score'] <=2:
        return 'NEW'
    elif row['R_Score'] <=3 and row['F_Score'] <=2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segmentation'] = rfm.apply(segment_customer,axis=1)
rfm['IsChurned'] = rfm['Recency']>90

# Merge & Export

In [15]:
df = df[
    (df['Customer_ID'].notna()) &
    (df['Quantity'] > 0)
].copy()

df['InvoiceMonth'] = pd.to_datetime(df['InvoiceDate']).dt.month_name()

df_final = df.merge(
    rfm[['Customer_ID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','Segmentation','IsChurned']], 
    on= 'Customer_ID',how='left'
)
df_final.to_excel('Ecom.xlsx',index=False)

In [16]:
print(df_final.shape)
df_final.head()

(779495, 18)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,SourceSheet,InvoiceMonth,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,Segmentation,IsChurned
0,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01,1.25,13085.0,United Kingdom,2009-2010,December,158,8,2433.28,3,2,2,Needs Attention,True
1,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01,5.95,13085.0,United Kingdom,2009-2010,December,158,8,2433.28,3,2,2,Needs Attention,True
2,489434,21871,SAVE THE PLANET MUG,24,2009-12-01,1.25,13085.0,United Kingdom,2009-2010,December,158,8,2433.28,3,2,2,Needs Attention,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01,2.10,13085.0,United Kingdom,2009-2010,December,158,8,2433.28,3,2,2,Needs Attention,True
4,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01,1.65,13085.0,United Kingdom,2009-2010,December,158,8,2433.28,3,2,2,Needs Attention,True
